In [44]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import random
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [45]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [46]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

def initLayer(i,o):
        weights = (torch.rand(i,o)  * 0.01).requires_grad_()
        bias = (torch.rand(o)  * 0.01).requires_grad_()
        return weights, bias


class NeuralNetwork(nn.Module):    
        def __init__(self):
                super().__init__()
                self.flatten = nn.Flatten()
                self.L1_weights, self.L1_bias = initLayer(28*28, 512)
                self.L2_weights, self.L2_bias = initLayer(512, 512)
                self.L3_weights, self.L3_bias = initLayer(512, 10)


        def linearReLUStack(self,x):
                x = x @ self.L1_weights + self.L1_bias
                x = torch.relu(x)
                x = x @ self.L2_weights + self.L2_bias
                x = torch.relu(x)
                x = x @ self.L3_weights + self.L3_bias
                return x

        def forward(self,x):
                x = self.flatten(x)
                logits = self.linearReLUStack(x)
                return logits
model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
)


In [47]:
def loss_function(pred, y):
    # pred: (batch_size, num_classes)
    # Uses cross-entropy loss

    exp_logits = pred.exp()
    probs = exp_logits / exp_logits.sum(dim=1, keepdim=True)

    correct_probs = probs[range(len(y)), y]
    loss = -(correct_probs.log()).mean()

    return loss

def optimizer_step(model, lr = 1e-2):
    with torch.no_grad():
        params= [model.L1_weights,model.L1_bias, model.L2_weights, model.L2_bias, model.L3_weights, model.L3_bias]
        for p in params:
            p -= lr * p.grad
            p.grad.zero_()

def train(dataloader,model: NeuralNetwork):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_function(pred, y)

        loss.backward()
        optimizer_step(model)

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


In [48]:
def test(dataloader, model : NeuralNetwork):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_function(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [49]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model)
    test(test_dataloader, model)
print("Done!")

Epoch 1
-------------------------------
loss: 2.297910  [   64/60000]
loss: 2.267036  [ 6464/60000]
loss: 2.428262  [12864/60000]
loss: 2.300324  [19264/60000]
loss: 2.312391  [25664/60000]
loss: 2.228431  [32064/60000]
loss: 2.277888  [38464/60000]
loss: 2.179098  [44864/60000]
loss: 2.255488  [51264/60000]
loss: 2.222618  [57664/60000]
Test Error: 
 Accuracy: 13.9%, Avg loss: 2.201700 

Epoch 2
-------------------------------
loss: 2.224719  [   64/60000]
loss: 2.198418  [ 6464/60000]
loss: 2.183584  [12864/60000]
loss: 2.158873  [19264/60000]
loss: 2.081321  [25664/60000]
loss: 2.028841  [32064/60000]
loss: 1.996727  [38464/60000]
loss: 1.819516  [44864/60000]
loss: 1.676024  [51264/60000]
loss: 1.481356  [57664/60000]
Test Error: 
 Accuracy: 45.4%, Avg loss: 1.396767 

Epoch 3
-------------------------------
loss: 1.498769  [   64/60000]
loss: 1.373975  [ 6464/60000]
loss: 1.173489  [12864/60000]
loss: 1.250360  [19264/60000]
loss: 1.064597  [25664/60000]
loss: 1.148836  [32064/600